In [1]:
# -*- coding: utf-8 -*-
# 【CP3-01 内存检查点】InMemorySaver + thread_id 实现多轮记忆
# 文件：CP3/01_in_memory.ipynb
# 作用：本 Cell 演示 【CP3-01 内存检查点】InMemorySaver + thread_id 实现多轮记忆 的完整可运行示例
# 阅读顺序：状态定义 → 节点定义 → 图构建 → 编译执行 → 结果观察

from dotenv import  load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver  # 内存检查点：进程内字典存储，适合开发/测试
from langgraph.graph import MessagesState, StateGraph,START,END

load_dotenv(override = True)

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body=
    {
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 声明状态
class OverAllState(MessagesState):
    output:str

#2. 声明节点
def llm_mode(state:OverAllState) -> OverAllState:
    messages = state["messages"]
    res = model.invoke(messages)
    return {
        "messages":[res]
    }

def output_node(state:OverAllState) -> OverAllState:
    return {
        "output":state["messages"][-1].content
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)  # 创建状态图构建器：绑定状态 Schema
builder.add_node("llm_node",llm_mode)  # 注册节点到图中
builder.add_node("output_node",output_node)  # 注册节点到图中
builder.add_edge(START,"llm_node")  # 起点扇出
builder.add_edge("llm_node","output_node")
builder.add_edge("output_node",END)  # 汇入终点

#4. 配置检查点存储器
checkpointer = InMemorySaver()  # 内存检查点：进程内字典存储，适合开发/测试
graph = builder.compile(checkpointer=checkpointer)  # 实例化检查点保存器

#5. 使用的时候必须填写线程ID
config = {
    "configurable":{
        "thread_id":"chapter03-01"  # 会话标识：同 thread_id 的调用共享同一条记忆链
    }
}

#6. 执行图
graph.invoke({"messages":[HumanMessage("你好,我是老王")]},config=config)  # 触发图执行：传入初始 State + config


{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='80ab1b4f-92e1-4f4b-85f3-befc7dafc41d'),
  AIMessage(content='你好老王！👋 很高兴见到你！今天有什么我可以帮你的吗？无论是闲聊、解答问题，还是需要一些建议，我都在这儿呢！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 8, 'total_tokens': 40, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'c6617f87-b1c0-4748-8a78-8cd0f6d5b1ba', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a02fcc-c473-7df2-8e93-9bb88118ecc5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 32, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})],
 'output': '你好老王！👋 很高兴见到你！今天有什么我

In [2]:
# -*- coding: utf-8 -*-
# 【CP3-01 内存检查点】InMemorySaver + thread_id 实现多轮记忆
# 文件：CP3/01_in_memory.ipynb
# 作用：本 Cell 演示 【CP3-01 内存检查点】InMemorySaver + thread_id 实现多轮记忆 的完整可运行示例
# 阅读顺序：状态定义 → 节点定义 → 图构建 → 编译执行 → 结果观察

graph.invoke({"messages":[HumanMessage("你好,我是谁")]},config=config)  # 触发图执行：传入初始 State + config


{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='80ab1b4f-92e1-4f4b-85f3-befc7dafc41d'),
  AIMessage(content='你好老王！👋 很高兴见到你！今天有什么我可以帮你的吗？无论是闲聊、解答问题，还是需要一些建议，我都在这儿呢！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 8, 'total_tokens': 40, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'c6617f87-b1c0-4748-8a78-8cd0f6d5b1ba', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a02fcc-c473-7df2-8e93-9bb88118ecc5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 32, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  HumanMessage(content='你好,我是谁', 

In [3]:
# -*- coding: utf-8 -*-
# 【CP3-01 内存检查点】InMemorySaver + thread_id 实现多轮记忆
# 文件：CP3/01_in_memory.ipynb
# 作用：本 Cell 演示 【CP3-01 内存检查点】InMemorySaver + thread_id 实现多轮记忆 的完整可运行示例
# 阅读顺序：状态定义 → 节点定义 → 图构建 → 编译执行 → 结果观察

config1 = {
    "configurable":{
        "thread_id":"chapter03-01xx"  # 会话标识：同 thread_id 的调用共享同一条记忆链
    }
}
graph.invoke({"messages":[HumanMessage("你好,我是谁")]},config=config1)  # 触发图执行：传入初始 State + config


{'messages': [HumanMessage(content='你好,我是谁', additional_kwargs={}, response_metadata={}, id='58b222e5-fa11-48bc-96fe-4cb5daf0317c'),
  AIMessage(content='你好！从我们的对话来看，我目前还不知道你的具体身份呢。你可以告诉我你的名字，或者想让我怎么称呼你吗？😊 无论你是谁，我都乐意为你提供帮助！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 8, 'total_tokens': 48, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'e4e45449-0a6d-4a1e-bec7-ff337b0a4ef6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a02fdc-e3cd-7261-9493-edaa88d3cf0f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 40, 'total_tokens': 48, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})],
 'output': '你好！从我们